# Normalize Data
Detects lined vs blank pages from PDF, crops lines to **Final Data**, then normalizes all output to **FinalDataNormalized**.

## Imports

In [120]:
import warnings
warnings.filterwarnings('ignore')
import cv2
import numpy as np
import os
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import fitz
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d

## Paths

In [121]:
INPUT_PDF_DIR        = "whole image"
FINAL_DATA_DIR       = "Final Data"
FINAL_DATA_NORM_DIR  = "FinalDataNormalized"

SKIP_TOP          = 20   # px to skip after printed line
PAD_BOTTOM        = 20   # px to add below next printed line
COLUMN_TOKEN      = "_COLUMNS"
SIGNATURE_TOKEN   = "_signature"

FINAL_SEGMENTATION_DIR = "FinalSegmentation"

os.makedirs(FINAL_DATA_DIR,         exist_ok=True)
os.makedirs(FINAL_DATA_NORM_DIR,    exist_ok=True)
os.makedirs(FINAL_SEGMENTATION_DIR, exist_ok=True)

## PDF Loading & Page Detection

In [122]:
def pdf_to_bgr_pages(path, dpi=300):
    doc   = fitz.open(path)
    scale = dpi / 72
    mat   = fitz.Matrix(scale, scale)
    pages = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR)
        pages.append(img)
    return pages


def horizontal_line_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    h, w = binary.shape
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(1, w // 5), 1))
    lines_only = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return cv2.countNonZero(lines_only)


def select_pages(path, dpi=300):
    """Returns (lined_page, blank_page). blank_page is None if only 1 page."""
    pages = pdf_to_bgr_pages(path, dpi=dpi)
    if len(pages) == 1:
        print("  Single-page PDF — treating as lined, no blank page")
        return pages[0], None
    scores = [horizontal_line_score(p) for p in pages]
    lined_idx = int(np.argmax(scores))
    blank_idx = 1 - lined_idx
    print(f"  Line scores: {scores}  ->  lined=page {lined_idx+1}, blank=page {blank_idx+1}")
    return pages[lined_idx], pages[blank_idx]

## Crop Lines (Lined Page)

In [123]:
def detect_printed_lines(img, debug=False):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    row_sums = np.zeros(h, dtype=float)
    for kw in [w // 12, w // 8, w // 5]:
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, 1))
        mask = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
        row_sums += mask.sum(axis=1).astype(float)

    row_sums = uniform_filter1d(row_sums, size=5)

    min_dist  = h // 50
    threshold = row_sums.max() * 0.08
    peaks, _  = find_peaks(row_sums, height=threshold, distance=min_dist)

    if debug:
        fig, axes = plt.subplots(1, 2, figsize=(16, 10))
        vis = img.copy()
        for y in peaks:
            cv2.line(vis, (0, int(y)), (w, int(y)), (0, 0, 255), 2)
        axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Detected {len(peaks)} lines (red)")
        axes[0].axis("off")
        axes[1].plot(row_sums)
        axes[1].axhline(threshold, color="r", linestyle="--", label="threshold")
        for y in peaks:
            axes[1].axvline(y, color="g", alpha=0.5)
        axes[1].set_title("Row sum profile (green = detected lines)")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

    return sorted(peaks.tolist())


def remove_corner_marks(img, corner_h_ratio=0.06, corner_w_ratio=0.08, min_area=150):
    """Erase L-shaped registration marks from all four corners of the page."""
    result = img.copy()
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    ch = int(h * corner_h_ratio)
    cw = int(w * corner_w_ratio)

    corners = [
        (0,      ch,      0,      cw),   # top-left
        (0,      ch,      w - cw, w),    # top-right
        (h - ch, h,       0,      cw),   # bottom-left
        (h - ch, h,       w - cw, w),    # bottom-right
    ]

    for ry1, ry2, cx1, cx2 in corners:
        roi_bin = np.zeros_like(binary)
        roi_bin[ry1:ry2, cx1:cx2] = binary[ry1:ry2, cx1:cx2]

        n, labels, stats, _ = cv2.connectedComponentsWithStats(roi_bin, connectivity=8)
        for lbl in range(1, n):
            if stats[lbl, cv2.CC_STAT_AREA] >= min_area:
                bx = stats[lbl, cv2.CC_STAT_LEFT]
                by = stats[lbl, cv2.CC_STAT_TOP]
                bw = stats[lbl, cv2.CC_STAT_WIDTH]
                bh = stats[lbl, cv2.CC_STAT_HEIGHT]
                result[by:by + bh, bx:bx + bw] = 255

    return result


def crop_lines(img, line_ys, out_dir, basename,
               skip_top=SKIP_TOP, pad_bottom=PAD_BOTTOM, y_max=None):
    """Crop one writing strip per detected baseline (N baselines → N total crops).

    line_01 uses the median inter-baseline spacing so it matches every other crop
    in height, even if the gap above the first baseline differs from later gaps.
    Signature uses the same interval logic as regular lines.

      line_01 : (line_ys[0] - median_spacing + skip_top) → line_ys[0] + pad_bottom
      line_02 : line_ys[0] + skip_top  → line_ys[1] + pad_bottom
      …
      line_N-1: line_ys[N-3]+skip_top → line_ys[N-2]+pad_bottom
      signature: line_ys[N-2]+skip_top → line_ys[N-1]+pad_bottom
    """
    h, w = img.shape[:2]
    if y_max is None:
        y_max = h
    saved = []

    # Median spacing across all regular writing intervals (excludes signature gap)
    spacings = [line_ys[i + 1] - line_ys[i] for i in range(len(line_ys) - 2)]
    median_spacing = int(np.median(spacings))

    # line_01: mirror the regular formula using a virtual baseline one spacing above
    y1_first = max(0, line_ys[0] - median_spacing + skip_top)
    y2_first = min(h, line_ys[0] + pad_bottom)
    name = f"{basename}_line_01.png"
    path = os.path.join(out_dir, name)
    cv2.imwrite(path, img[y1_first:y2_first, :])
    saved.append(path)

    # line_02 … line_N-1: intervals between consecutive baselines (all but last)
    for i in range(len(line_ys) - 2):
        y1 = min(h, line_ys[i]     + skip_top)
        y2 = min(h, line_ys[i + 1] + pad_bottom)
        name = f"{basename}_line_{i + 2:02d}.png"
        path = os.path.join(out_dir, name)
        cv2.imwrite(path, img[y1:y2, :])
        saved.append(path)

    # Signature: same interval logic — writing above the short signature rule
    y1_sig = min(h, line_ys[-2] + skip_top)
    y2_sig = min(h, line_ys[-1] + pad_bottom)
    if y2_sig > y1_sig + 5:
        name = f"{basename}{SIGNATURE_TOKEN}.png"
        path = os.path.join(out_dir, name)
        cv2.imwrite(path, img[y1_sig:y2_sig, :])
        saved.append(path)

    return saved

## Column Detection (Blank Page)

In [124]:
def detect_frame_bounds(img):
    """Detect the inner bounds of the printed frame rectangle.
    Returns (x_left, y_top, x_right, y_bottom) just inside the frame lines."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 5, 1))
    h_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)
    row_sums = uniform_filter1d(h_lines.sum(axis=1).astype(float), size=5)
    h_peaks, _ = find_peaks(row_sums, height=row_sums.max() * 0.3, distance=h // 10)

    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 5))
    v_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel)
    col_sums = uniform_filter1d(v_lines.sum(axis=0).astype(float), size=5)
    v_peaks, _ = find_peaks(col_sums, height=col_sums.max() * 0.3, distance=w // 10)

    margin = 15
    y_top    = int(h_peaks[0])  + margin if len(h_peaks) >= 2 else 0
    y_bottom = int(h_peaks[-1]) - margin if len(h_peaks) >= 2 else h
    x_left   = int(v_peaks[0])  + margin if len(v_peaks) >= 2 else 0
    x_right  = int(v_peaks[-1]) - margin if len(v_peaks) >= 2 else w

    print(f"  Frame inner bounds: x=[{x_left}, {x_right}], y=[{y_top}, {y_bottom}]")
    return x_left, y_top, x_right, y_bottom


def _bands_from_profile(profile, threshold_ratio, min_gap):
    """Find content bands in a 1-D sum profile."""
    thresh = profile.max() * threshold_ratio
    in_band = profile > thresh

    raw, start = [], None
    for i, val in enumerate(in_band):
        if val and start is None:
            start = i
        elif not val and start is not None:
            raw.append([start, i])
            start = None
    if start is not None:
        raw.append([start, len(profile)])

    merged = []
    for b in raw:
        if merged and b[0] - merged[-1][1] < min_gap:
            merged[-1][1] = b[1]
        else:
            merged.append(b)
    return merged


def _filter_isolated_bands(bands):
    """Remove bands separated from the main cluster by a gap much larger than
    the typical inter-band spacing (catches signatures, labels, etc.)."""
    if len(bands) < 2:
        return bands

    gaps = [bands[i][0] - bands[i - 1][1] for i in range(1, len(bands))]
    median_gap = float(np.median(gaps))
    max_gap    = max(median_gap * 4, 20)

    kept = [bands[0]]
    for i in range(1, len(bands)):
        if (bands[i][0] - kept[-1][1]) <= max_gap:
            kept.append(bands[i])
    return kept


def column_detection(img, out_dir, basename):
    """Detect columns and individual number rows inside the frame.

    Returns (col_boxes, row_boxes) where:
      col_boxes — list of (x1, y1, x2, y2) in full-image coords, one per column
      row_boxes — list of (col_idx, row_idx, x1, y1, x2, y2) one per number
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    fx1, fy1, fx2, fy2 = detect_frame_bounds(img)
    gray_inner = gray[fy1:fy2, fx1:fx2]
    hi, wi = gray_inner.shape

    _, binary = cv2.threshold(gray_inner, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Remove blobs touching inner-region border (residual frame artefacts)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    clean = binary.copy()
    for lbl in range(1, n):
        bx = stats[lbl, cv2.CC_STAT_LEFT];  by = stats[lbl, cv2.CC_STAT_TOP]
        bw = stats[lbl, cv2.CC_STAT_WIDTH]; bh = stats[lbl, cv2.CC_STAT_HEIGHT]
        if bx == 0 or by == 0 or bx + bw >= wi or by + bh >= hi:
            clean[labels == lbl] = 0

    # wi//80 smoothing: sharp enough to preserve narrow inter-column gaps
    col_profile = uniform_filter1d(clean.sum(axis=0).astype(float), size=max(1, wi // 80))
    # Low threshold (2%) so italic / lighter handwriting still registers
    raw_bands = _bands_from_profile(col_profile, threshold_ratio=0.02, min_gap=max(5, wi // 100))
    # Reject bands narrower than 25 px — those are frame artefacts, not number columns
    col_bands = [b for b in raw_bands if (b[1] - b[0]) >= 25]

    vis       = img.copy()
    col_boxes = []
    row_boxes = []

    min_col_height = hi // 5

    real_col_idx = 0
    for cx1, cx2 in col_bands:
        strip = clean[:, cx1:cx2]

        # Minimal smoothing + tiny min_gap: each number stays as its own band so
        # the large gap to any stray content (signature) is an obvious outlier.
        row_prof_fine = uniform_filter1d(strip.sum(axis=1).astype(float), size=3)
        y_bands = _bands_from_profile(row_prof_fine, threshold_ratio=0.03, min_gap=3)
        y_bands = _filter_isolated_bands(y_bands)
        if not y_bands:
            continue
        cy1, cy2 = y_bands[0][0], y_bands[-1][1]

        if (cy2 - cy1) < min_col_height:
            continue

        real_col_idx += 1
        abs_cx1, abs_cy1 = fx1 + cx1, fy1 + cy1
        abs_cx2, abs_cy2 = fx1 + cx2, fy1 + cy2
        col_boxes.append((abs_cx1, abs_cy1, abs_cx2, abs_cy2))
        cv2.rectangle(vis, (abs_cx1, abs_cy1), (abs_cx2, abs_cy2), (0, 0, 255), 3)
        cv2.putText(vis, f"col {real_col_idx}", (abs_cx1 + 4, abs_cy1 + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

        # ── Segment individual numbers (data only, not drawn) ──
        col_clean = clean[cy1:cy2, cx1:cx2]
        rh = cy2 - cy1
        row_prof2 = uniform_filter1d(col_clean.sum(axis=1).astype(float), size=3)
        num_bands = _bands_from_profile(row_prof2, threshold_ratio=0.04,
                                        min_gap=max(3, rh // 40))
        num_bands = _filter_isolated_bands(num_bands)

        for r_idx, (ry1, ry2) in enumerate(num_bands):
            num_strip = col_clean[ry1:ry2, :]
            cx_prof   = num_strip.sum(axis=0).astype(float)
            xs = np.where(cx_prof > 0)[0]
            rx1 = int(xs[0])  if len(xs) else 0
            rx2 = int(xs[-1]) if len(xs) else (cx2 - cx1)

            ax1 = fx1 + cx1 + rx1;  ay1 = fy1 + cy1 + ry1
            ax2 = fx1 + cx1 + rx2;  ay2 = fy1 + cy1 + ry2
            row_boxes.append((real_col_idx, r_idx + 1, ax1, ay1, ax2, ay2))

        print(f"  col {real_col_idx}: {len(num_bands)} number(s) detected")

    out_name = f"{basename}{COLUMN_TOKEN}.png"
    cv2.imwrite(os.path.join(out_dir, out_name), vis)
    print(f"  {len(col_boxes)} column(s) total -> saved {out_name}")
    return col_boxes, row_boxes


In [125]:
def detect_and_crop_frame(img):
    """Detect the printed rectangular frame and return the image cropped to it.

    Looks for strong horizontal lines (top/bottom frame edges) and strong vertical
    lines (left/right frame edges). All four sides must be found for a valid frame.

    Returns:
        Cropped image (everything outside the frame removed) if frame is fully detected.
        None if the frame cannot be fully identified — caller should skip the page.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
    h, w = gray.shape

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Long horizontal runs → top and bottom frame lines
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 5, 1))
    h_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)
    row_sums = uniform_filter1d(h_lines.sum(axis=1).astype(float), size=5)
    if row_sums.max() == 0:
        print("  No horizontal frame lines found — skipping page")
        return None
    h_peaks, _ = find_peaks(row_sums, height=row_sums.max() * 0.3, distance=h // 10)

    # Long vertical runs → left and right frame lines
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 5))
    v_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel)
    col_sums = uniform_filter1d(v_lines.sum(axis=0).astype(float), size=5)
    if col_sums.max() == 0:
        print("  No vertical frame lines found — skipping page")
        return None
    v_peaks, _ = find_peaks(col_sums, height=col_sums.max() * 0.3, distance=w // 10)

    # Need at least top + bottom horizontal and left + right vertical
    if len(h_peaks) < 2 or len(v_peaks) < 2:
        print(f"  Frame incomplete (h_peaks={len(h_peaks)}, v_peaks={len(v_peaks)}) — skipping page")
        return None

    y_top,  y_bottom = int(h_peaks[0]),  int(h_peaks[-1])
    x_left, x_right  = int(v_peaks[0]),  int(v_peaks[-1])

    # Sanity: frame must span at least half the page in each direction
    if (y_bottom - y_top) < h * 0.5 or (x_right - x_left) < w * 0.5:
        print(f"  Frame too small (y={y_top}–{y_bottom}, x={x_left}–{x_right}) — skipping page")
        return None

    print(f"  Frame detected: x=[{x_left}, {x_right}], y=[{y_top}, {y_bottom}]")
    return img[y_top:y_bottom, x_left:x_right]

In [126]:
def crop_to_writing_area(img):
    """Trim the top corner bracket marks from a frame-cropped lined page.

    Finds the bottommost extent of the top-left and top-right corner marks and
    returns img[y_top:, :] — full width, full bottom height preserved so the
    signature line (which sits at the same level as the bottom corner marks) is
    not accidentally cut off.

    Falls back to the original image if marks cannot be found.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
    h, w = gray.shape

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Search in the outer 15 % of height and 15 % of width for top corner marks
    ch = int(h * 0.15)
    cw = int(w * 0.15)
    min_area = (ch * cw) * 0.003

    def bottom_of(region, off_y):
        n, _, stats, _ = cv2.connectedComponentsWithStats(region, connectivity=8)
        valid = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area]
        if not valid:
            return None
        return max(stats[i, cv2.CC_STAT_TOP] + stats[i, cv2.CC_STAT_HEIGHT] for i in valid) + off_y

    tl_bot = bottom_of(binary[:ch, :cw],   0)
    tr_bot = bottom_of(binary[:ch, w-cw:], 0)

    top_vals = [v for v in [tl_bot, tr_bot] if v is not None]
    if not top_vals:
        return img

    y_top = max(top_vals)

    # Sanity: don't trim more than 20 % of the image
    if y_top < 1 or y_top > h * 0.2:
        return img

    print(f"  Trimmed top corner marks: y_top={y_top}")
    return img[y_top:, :]

## Image Normalization

In [127]:
def binarize_image(img: np.ndarray, method: str = "adaptive") -> np.ndarray:
    if method == "otsu":
        _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif method == "adaptive":
        binary = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 10)
    elif method == "sauvola":
        binary = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 25, 15)
    else:
        raise ValueError(f"Unknown binarization method: {method}")
    return binary


def remove_noise(binary_img: np.ndarray, min_component_size: int = 10) -> np.ndarray:
    kernel = np.ones((2, 2), np.uint8)
    opened = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel, iterations=1)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(255 - opened, connectivity=8)
    output = np.zeros_like(opened)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_component_size:
            output[labels == i] = 255
    return 255 - output


def crop_to_content(img: np.ndarray, padding: int = 20) -> np.ndarray:
    coords = cv2.findNonZero(255 - img)
    if coords is None:
        return img
    x, y, w, h = cv2.boundingRect(coords)
    # Only crop vertically — left/right borders stay at full image width
    y = max(0, y - padding)
    h = min(img.shape[0] - y, h + 2 * padding)
    return img[y:y+h, :]


def normalize_height(img: np.ndarray, target_height: int = 64) -> np.ndarray:
    h, w = img.shape[:2]
    new_width = int(target_height * (w / h))
    return cv2.resize(img, (new_width, target_height), interpolation=cv2.INTER_AREA)


def normalize_image(img_bgr: np.ndarray,
                    binarize: bool = True, denoise: bool = True,
                    crop: bool = True, normalize_size: bool = False,
                    target_height: int = 64) -> np.ndarray:
    """Run full normalization pipeline on an in-memory BGR image; returns grayscale."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if len(img_bgr.shape) == 3 else img_bgr.copy()
    if binarize:
        gray = binarize_image(gray, method="adaptive")
    if denoise:
        gray = remove_noise(gray)
    if crop:
        gray = crop_to_content(gray)
    if normalize_size:
        gray = normalize_height(gray, target_height=target_height)
    return gray

## Word Segmentation

In [128]:
SEG_MIN_COMP_H = 5
SEG_MIN_COMP_W = 4
SEG_MIN_WORD_W = 15
SEG_GAP_MULT   = 1.9
SEG_PADDING    = 4
SEG_TOP_MARGIN = 25  # px to ignore at top of crop (bleed-through from previous line)


def _seg_get_comps(binary):
    if binary.shape[0] < 5:
        return []
    num_labels, _labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    comps = []
    for i in range(1, num_labels):
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        if w >= SEG_MIN_COMP_W and h >= SEG_MIN_COMP_H:
            comps.append((x, y, w, h))
    return comps


def _seg_median_gap(comps):
    if len(comps) < 2:
        return 20
    gaps = [comps[i][0] - (comps[i-1][0] + comps[i-1][2]) for i in range(1, len(comps))]
    pos_gaps = [g for g in gaps if g > 0]
    return float(np.median(pos_gaps)) * SEG_GAP_MULT if pos_gaps else 20


def _seg_merge_comps(comps):
    if not comps:
        return []
    comps = sorted(comps, key=lambda b: b[0])
    thresh = _seg_median_gap(comps)
    merged = []
    cx, cy, cw, ch = comps[0]
    for (x, y, w, h) in comps[1:]:
        if x - (cx + cw) <= thresh:
            right  = max(cx + cw, x + w)
            bottom = max(cy + ch, y + h)
            cx, cy = min(cx, x), min(cy, y)
            cw, ch = right - cx, bottom - cy
        else:
            if cw >= SEG_MIN_WORD_W:
                merged.append((cx, cy, cw, ch))
            cx, cy, cw, ch = x, y, w, h
    if cw >= SEG_MIN_WORD_W:
        merged.append((cx, cy, cw, ch))
    return merged


def get_word_boxes(img_bgr):
    """Return list of (x, y, w, h) word bounding boxes for a line image.

    Detects the printed baseline, blanks it and everything below, then blanks
    the top margin to exclude bleed-through from the previous line.
    Final boxes are clipped to [SEG_TOP_MARGIN, baseline_y].
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Detect and erase the printed baseline (long horizontal run)
    kernel_w = max(30, w // 6)
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_w, 1))
    h_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)
    row_sums = h_lines.sum(axis=1)
    baseline_y = h
    if row_sums.max() > 0:
        baseline_y = int(np.argmax(row_sums))
        binary[max(0, baseline_y - 5):, :] = 0
    binary[:SEG_TOP_MARGIN, :] = 0

    comps = _seg_get_comps(binary)
    # Only drop components that start inside the top-margin bleed-through zone
    comps = [(x, y, cw, ch) for (x, y, cw, ch) in comps if y >= SEG_TOP_MARGIN]

    boxes = _seg_merge_comps(comps)

    # Clip merged boxes to the writing zone
    result = []
    for (x, y, bw, bh) in boxes:
        y1 = max(y, SEG_TOP_MARGIN)
        y2 = min(y + bh, baseline_y)
        if y2 - y1 >= SEG_MIN_COMP_H:
            result.append((x, y1, bw, y2 - y1))
    return result


def annotate_segmentation(normalized_gray, seg_dir, filename):
    """Draw green word-boxes on a copy of the normalized grayscale image
    and save to seg_dir with the same filename."""
    vis = cv2.cvtColor(normalized_gray, cv2.COLOR_GRAY2BGR)
    h_img, w_img = vis.shape[:2]
    boxes = get_word_boxes(vis)
    for (x, y, w, h) in boxes:
        x1 = max(x - SEG_PADDING, 0)
        y1 = max(y - SEG_PADDING, 0)
        x2 = min(x + w + SEG_PADDING, w_img - 1)
        y2 = min(y + h + SEG_PADDING, h_img - 1)
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.imwrite(os.path.join(seg_dir, filename), vis)
    return len(boxes)

## Run Full Pipeline

In [129]:
# pdf_files = sorted(glob.glob(os.path.join(INPUT_PDF_DIR, "*.pdf")))
# if not pdf_files:
#     raise FileNotFoundError(f"No PDF files found in {INPUT_PDF_DIR!r}")

# for pdf_path in pdf_files:
#     base = os.path.splitext(os.path.basename(pdf_path))[0]
#     print(f"Processing: {pdf_path}")

#     lined_page, blank_page = select_pages(pdf_path)

#     # ── Lined page: crop -> Final Data, normalize -> FinalDataNormalized ──
#     print(f"  Lined page: {lined_page.shape[1]}x{lined_page.shape[0]} px")
#     line_ys  = detect_printed_lines(lined_page, debug=False)
#     x_bounds = detect_content_x_bounds(lined_page)
#     print(f"  Found {len(line_ys)} printed lines -> {len(line_ys)-1} crops")

#     if len(line_ys) < 2:
#         print("  Not enough lines detected — skipping lined page")
#     else:
#         crop_paths = crop_lines(lined_page, line_ys, FINAL_DATA_DIR, base, x_bounds=x_bounds)
#         print(f"  Saved {len(crop_paths)} line crops to {FINAL_DATA_DIR!r}")

#         for crop_path in crop_paths:
#             crop_img  = cv2.imread(crop_path)
#             normalized = normalize_image(crop_img)
#             out_path  = os.path.join(FINAL_DATA_NORM_DIR, os.path.basename(crop_path))
#             cv2.imwrite(out_path, normalized)

#         print(f"  Normalized {len(crop_paths)} crops -> {FINAL_DATA_NORM_DIR!r}")

#     # ── Blank page: normalize then column detection -> FinalDataNormalized ──
#     if blank_page is not None:
#         print(f"  Blank page: {blank_page.shape[1]}x{blank_page.shape[0]} px")
#         norm_gray = normalize_image(blank_page, normalize_size=False)
#         norm_bgr  = cv2.cvtColor(norm_gray, cv2.COLOR_GRAY2BGR)
#         column_detection(norm_bgr, FINAL_DATA_NORM_DIR, base)

## Process All PDFs (Batch Loop)

In [130]:
def process_pdf(pdf_path):
    base = os.path.splitext(os.path.basename(pdf_path))[0]
    print(f"\nProcessing: {pdf_path}")

    lined_page, blank_page = select_pages(pdf_path)

    # ── Lined page ──
    # Step 1: crop to the outer frame (removes page margin noise)
    lined_frame = detect_and_crop_frame(lined_page)
    if lined_frame is None:
        print("  Lined page frame not detected — skipping lined page")
    else:
        # Step 2: erase corner bracket marks from all 4 corners by painting them white.
        # corner_h_ratio=0.18 because the L-shaped marks extend ~15 % from each edge.
        lined_cropped = remove_corner_marks(lined_frame, corner_h_ratio=0.18, corner_w_ratio=0.10)
        inner_h = lined_cropped.shape[0]

        line_ys = detect_printed_lines(lined_cropped, debug=False)
        print(f"  Found {len(line_ys)} printed lines -> {len(line_ys)-1} writing crops + 1 signature crop")

        if len(line_ys) >= 2:
            crop_paths = crop_lines(lined_cropped, line_ys, FINAL_DATA_DIR, base, y_max=inner_h)
            for crop_path in crop_paths:
                crop_img   = cv2.imread(crop_path)
                normalized = normalize_image(crop_img)
                out_path   = os.path.join(FINAL_DATA_NORM_DIR, os.path.basename(crop_path))
                cv2.imwrite(out_path, normalized)
            print(f"  Saved {len(crop_paths)} crops -> {FINAL_DATA_DIR!r} and normalized -> {FINAL_DATA_NORM_DIR!r}")

            for crop_path in crop_paths:
                norm_path = os.path.join(FINAL_DATA_NORM_DIR, os.path.basename(crop_path))
                norm_img  = cv2.imread(norm_path, cv2.IMREAD_GRAYSCALE)
                if norm_img is not None:
                    annotate_segmentation(norm_img, FINAL_SEGMENTATION_DIR, os.path.basename(crop_path))
            print(f"  Annotated {len(crop_paths)} segmentation images -> {FINAL_SEGMENTATION_DIR!r}")
        else:
            print("  Not enough lines detected — skipping lined page")

    # ── Blank page: full frame crop then column detection ──
    if blank_page is not None:
        blank_cropped = detect_and_crop_frame(blank_page)
        if blank_cropped is None:
            print("  Blank page frame not detected — skipping column detection")
        else:
            norm_gray = normalize_image(blank_cropped, normalize_size=False)
            norm_bgr  = cv2.cvtColor(norm_gray, cv2.COLOR_GRAY2BGR)
            column_detection(norm_bgr, FINAL_DATA_NORM_DIR, base)


# ── Run over every PDF in the whole image folder ──
pdf_files = sorted(glob.glob(os.path.join(INPUT_PDF_DIR, "*.pdf")))
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {INPUT_PDF_DIR!r}")

print(f"Found {len(pdf_files)} PDF(s) in {INPUT_PDF_DIR!r}")
for pdf_path in pdf_files:
    process_pdf(pdf_path)

print("\nDone.")

Found 16 PDF(s) in 'whole image'

Processing: whole image\SaraRosenboum.pdf
  Line scores: [155699, 56989]  ->  lined=page 1, blank=page 2
  Frame detected: x=[88, 2476], y=[32, 3411]
  Found 21 printed lines -> 20 writing crops + 1 signature crop
  Saved 21 crops -> 'Final Data' and normalized -> 'FinalDataNormalized'
  Annotated 21 segmentation images -> 'FinalSegmentation'
  Frame detected: x=[97, 2486], y=[34, 3418]
  Frame inner bounds: x=[20, 2372], y=[20, 3365]
  col 1: 1 number(s) detected
  col 2: 1 number(s) detected
  col 3: 7 number(s) detected
  3 column(s) total -> saved SaraRosenboum_COLUMNS.png

Processing: whole image\Scan1.pdf
  Line scores: [178343, 51570]  ->  lined=page 1, blank=page 2
  Frame detected: x=[114, 2406], y=[61, 3357]
  Found 21 printed lines -> 20 writing crops + 1 signature crop
  Saved 21 crops -> 'Final Data' and normalized -> 'FinalDataNormalized'
  Annotated 21 segmentation images -> 'FinalSegmentation'
  Frame detected: x=[131, 2415], y=[132, 34

In [131]:
# Delete line images with 0 detected word boxes from both FinalSegmentation and FinalDataNormalized
deleted = 0
seg_files = sorted(f for f in os.listdir(FINAL_SEGMENTATION_DIR) if f.endswith(".png"))
for fname in seg_files:
    seg_path  = os.path.join(FINAL_SEGMENTATION_DIR, fname)
    norm_path = os.path.join(FINAL_DATA_NORM_DIR,    fname)
    seg_img = cv2.imread(seg_path)
    if seg_img is None:
        continue
    # Green rectangle pixels are exactly BGR (0, 200, 0)
    has_boxes = (
        (seg_img[:, :, 0] == 0)   &
        (seg_img[:, :, 1] == 200) &
        (seg_img[:, :, 2] == 0)
    ).any()
    if not has_boxes:
        os.remove(seg_path)
        if os.path.exists(norm_path):
            os.remove(norm_path)
        deleted += 1
        print(f"  Deleted (0 boxes): {fname}")
print(f"Done. Removed {deleted} / {len(seg_files)} images with no word boxes.")


  Deleted (0 boxes): scan12_line_01.png
  Deleted (0 boxes): scan12_signature.png
Done. Removed 2 / 336 images with no word boxes.
